In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
os.listdir("/content/drive/MyDrive/Gloves_det_Project/runs/ppe_glove-2/weights")

['last.pt', 'best.pt']

In [16]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import json

# Load model
model = YOLO("/content/drive/MyDrive/Gloves_det_Project/runs/ppe_glove-2/weights/best.pt")

input_dir = Path("/content/drive/MyDrive/Gloves_det_Project/inference/input")
output_dir = Path("/content/drive/MyDrive/Gloves_det_Project/inference/output")
output_dir.mkdir(exist_ok=True)

logs = []

for image_path in input_dir.glob("*.jpg"):

    results = model.predict(
        source=str(image_path),
        conf=0.5,
        verbose=False
    )[0]

    # Save annotated image
    cv2.imwrite(
        str(output_dir / image_path.name),
        results[0].plot()
    )

    detections = []

    for box in results.boxes:

        x1, y1, x2, y2 = box.xyxy[0].tolist()

        detections.append({
            "class": results.names[int(box.cls[0])],
            "confidence": round(float(box.conf[0]), 3),
            "bbox": [
                round(x1, 2),
                round(y1, 2),
                round(x2, 2),
                round(y2, 2)
            ]
        })

    logs.append({
        "image": image_path.name,
        "detections": detections
    })

with open(output_dir / "detection_logs.json", "w") as f:
    json.dump(logs, f, indent=4)

print("Inference Complete.")

Inference Complete.
